In [1]:
import pandas as pd

In [2]:
from google.colab import files

In [3]:
uploaded=files.upload()

Saving sales_raw_500.xlsx to sales_raw_500.xlsx


In [6]:
df=pd.read_excel('sales_raw_500.xlsx')
df

,Order_ID,Order_Date,Customer,Region,Product,Sales,Cost
0,1001,2023-01-13,Rohit,South,Mobile,23289,19118
1,1002,2023-10-07,Aman,West,Laptop,8905,5593
2,1003,2023-11-05,Rahul,South,Monitor,59987,39959
3,1004,2024-02-19,Rahul,South,Printer,49597,33892
4,1005,2024-01-26,Anjali,North,Laptop,54797,34468
...,...,...,...,...,...,...,...
515,1389,2024-02-12,Vikas,North,Laptop,17924,12145
516,1496,2023-04-24,Kiran,North,Printer,23434,16388
517,1031,2023-02-19,Vikas,East,Printer,58883,43579
518,1317,2023-07-05,Aman,North,Laptop,76781,66865


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 520 entries, 0 to 519
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Order_ID    520 non-null    int64         
 1   Order_Date  502 non-null    datetime64[ns]
 2   Customer    520 non-null    object        
 3   Region      520 non-null    object        
 4   Product     520 non-null    object        
 5   Sales       520 non-null    int64         
 6   Cost        520 non-null    int64         
dtypes: datetime64[ns](1), int64(3), object(3)
memory usage: 28.6+ KB


In [9]:
df.shape

(520, 7)

In [11]:
df.columns

Index(['Order_ID', 'Order_Date', 'Customer', 'Region', 'Product', 'Sales',
       'Cost'],
      dtype='object')

In [12]:
df.isnull().sum()

,0
Order_ID,0
Order_Date,18
Customer,0
Region,0
Product,0
Sales,0
Cost,0


In [14]:
df=df.dropna(subset=['Order_Date'])

In [16]:
df.duplicated()

,0
0,False
1,False
2,False
3,False
4,False
...,...
515,True
516,True
517,True
518,True


In [17]:
df=df.drop_duplicates()

#connect sql

In [18]:
import pandas as pd
import sqlite3

In [20]:
df = pd.read_excel("sales_raw_500.xlsx")

df.head()

,Order_ID,Order_Date,Customer,Region,Product,Sales,Cost
0,1001,2023-01-13,Rohit,South,Mobile,23289,19118
1,1002,2023-10-07,Aman,West,Laptop,8905,5593
2,1003,2023-11-05,Rahul,South,Monitor,59987,39959
3,1004,2024-02-19,Rahul,South,Printer,49597,33892
4,1005,2024-01-26,Anjali,North,Laptop,54797,34468


In [21]:
conn = sqlite3.connect("sales.db")

#import the data in sqlite

In [22]:
df.to_sql(
    "sales",
    conn,
    if_exists="replace",
    index=False
)

520

#check sqlite

In [23]:
query = """
SELECT *
FROM sales
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,Order_ID,Order_Date,Customer,Region,Product,Sales,Cost
0,1001,2023-01-13 00:00:00,Rohit,South,Mobile,23289,19118
1,1002,2023-10-07 00:00:00,Aman,West,Laptop,8905,5593
2,1003,2023-11-05 00:00:00,Rahul,South,Monitor,59987,39959
3,1004,2024-02-19 00:00:00,Rahul,South,Printer,49597,33892
4,1005,2024-01-26 00:00:00,Anjali,North,Laptop,54797,34468
5,1006,2023-11-06 00:00:00,Rohit,North,Printer,75284,47989
6,1007,2023-02-10 00:00:00,Suresh,East,Monitor,52400,40515
7,1008,2023-01-24 00:00:00,Priya,East,Laptop,35512,30538
8,1009,2023-08-21 00:00:00,Anjali,South,Tablet,51566,34180
9,1010,2023-12-16 00:00:00,Aman,South,Monitor,37087,24070


#total sales

In [24]:
query = """
SELECT SUM(Sales) AS Total_Sales
FROM sales;
"""

pd.read_sql_query(query, conn)

,Total_Sales
0,21387396


#total profit

In [25]:
query = """
SELECT
    SUM(Sales) AS Total_Sales,
    SUM(Cost) AS Total_Cost,
    SUM(Sales - Cost) AS Total_Profit
FROM sales;
"""

pd.read_sql_query(query, conn)

,Total_Sales,Total_Cost,Total_Profit
0,21387396,15951001,5436395


#regionwise sale

In [26]:
query = """
SELECT
    Region,
    SUM(Sales) AS Total_Sales
FROM sales
GROUP BY Region
ORDER BY Total_Sales DESC;
"""

pd.read_sql_query(query, conn)

,Region,Total_Sales
0,South,5660991
1,North,5614094
2,West,5057005
3,East,5055306


#Product-wise Sales

In [27]:
query = """
SELECT
    Product,
    SUM(Sales) AS Total_Sales,
    SUM(Sales - Cost) AS Total_Profit
FROM sales
GROUP BY Product
ORDER BY Total_Sales DESC;
"""

pd.read_sql_query(query, conn)

,Product,Total_Sales,Total_Profit
0,Printer,4819084,1231067
1,Tablet,4345390,1123323
2,Laptop,4226723,1038507
3,Mobile,4033522,1020764
4,Monitor,3962677,1022734


#customer-wise sales

In [28]:
query = """
SELECT
    Customer,
    SUM(Sales) AS Total_Sales,
    SUM(Sales - Cost) AS Total_Profit
FROM sales
GROUP BY Customer
ORDER BY Total_Sales DESC;
"""

pd.read_sql_query(query, conn)

,Customer,Total_Sales,Total_Profit
0,Suresh,2419253,636618
1,Vikas,2372448,609742
2,Rahul,2268599,593415
3,Pooja,2246575,542784
4,Kiran,2215308,554510
5,Rohit,2209377,531653
6,Neha,2195325,546139
7,Aman,1996728,531271
8,Anjali,1861817,454948
9,Priya,1601966,435315


#close database

In [29]:
conn.close()